# Tesseract OCR Benchmark (Latest — v5.x)

Benchmark **Tesseract 5.x** (latest stable) on the **Sinhala OCR LK Acts 1010** test dataset.

This notebook mirrors the structure of the LightOnOCR benchmark for direct comparison.

- **Model**: Tesseract OCR 5.x (LSTM engine) with Sinhala language pack (`sin`)
- **Dataset**: `avishadilhara/sinhala-ocr-lk-acts-1010` (202 test samples)
- **Metrics**: CER, WER, BLEU, ANLS, METEOR, Character Accuracy

In [ ]:
%%capture
# ========================================
# STEP 0: INSTALL TESSERACT 5.x (LATEST)
# ========================================
# Kaggle/Colab default repos ship Tesseract 4.x.
# We add the official PPA to get the latest Tesseract 5.x.

import subprocess, sys

# Add the Tesseract PPA for latest 5.x builds
!apt-get update -qq
!apt-get install -y -qq software-properties-common
!add-apt-repository -y ppa:alex-p/tesseract-ocr5
!apt-get update -qq

# Install Tesseract 5.x + Sinhala language data
!apt-get install -y -qq tesseract-ocr tesseract-ocr-sin

# Install Python packages
!pip install -q pytesseract
!pip install -q datasets
!pip install -q jiwer
!pip install -q python-Levenshtein
!pip install -q Pillow>=11.3.0
!pip install -q huggingface-hub>=1.3.1

In [ ]:
# ========================================
# STEP 1: IMPORTS & SETUP
# ========================================

import os
import json
import time
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import pickle
from PIL import Image
import pytesseract

# Metrics
from jiwer import cer, wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk

# Download NLTK data for METEOR (silently)
import warnings
warnings.filterwarnings('ignore')

try:
    nltk.data.find('wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('omw-1.4', quiet=True)

In [ ]:
# ========================================
# STEP 2: VERIFY TESSERACT 5.x INSTALLATION
# ========================================
print("=" * 80)
print(" TESSERACT OCR SETUP")
print("=" * 80)

# Print Tesseract version
tesseract_version = pytesseract.get_tesseract_version()
print(f"\n Tesseract Version: {tesseract_version}")

# Verify we got Tesseract 5.x
major_version = int(str(tesseract_version).split('.')[0])
if major_version >= 5:
    print(f" Tesseract 5.x confirmed (latest stable)")
else:
    print(f" Got Tesseract {major_version}.x — expected 5.x")
    print("    The PPA may not have applied. Results will use this version.")

# List available languages
available_langs = pytesseract.get_languages(config='')
print(f" Available Languages: {available_langs}")

# Verify Sinhala is available
if 'sin' in available_langs:
    print("  Sinhala language pack (sin) is installed")
else:
    print(" [ERROR] Sinhala language pack NOT found!")
    print("    Install it with: apt-get install tesseract-ocr-sin")
    raise RuntimeError("Sinhala language pack not available")

print(f"\n Tesseract binary: {pytesseract.pytesseract.tesseract_cmd}")

# Print full version string from CLI
!tesseract --version


In [ ]:
# ==================== LOAD FROM HUGGING FACE ====================

from datasets import load_dataset
from huggingface_hub import login

#  Login to Hugging Face (if private dataset)
login()  # Enter your token

In [ ]:
# ========================================
# STEP 3: LOAD TEST DATASET
# ========================================
print("\n" + "="*80)
print(" LOADING TEST DATASET")
print("="*80)

# Load your dataset
print("\nLoading dataset from Hugging Face...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f" Dataset loaded!")
print(f"   Test samples: {len(dataset['test'])}")

In [ ]:
# ========================================
# CONFIGURATION
# ========================================
CHECKPOINT_INTERVAL = 10  # Save checkpoint every N samples

# Tesseract configuration
TESSERACT_LANG = 'sin'           # Sinhala language
TESSERACT_PSM = 6                # Page segmentation mode: Assume a single uniform block of text
TESSERACT_OEM = 3                # OCR Engine Mode: Default (LSTM + Legacy)
TESSERACT_CONFIG = f'--oem {TESSERACT_OEM} --psm {TESSERACT_PSM}'

In [ ]:
# ========================================
# STEP 4: DEFINE METRICS FUNCTIONS
# ========================================

def calculate_anls(ground_truth, prediction, threshold=0.5):
    """Calculate Average Normalized Levenshtein Similarity (ANLS)"""
    try:
        from Levenshtein import distance as levenshtein_distance

        if len(ground_truth) == 0 and len(prediction) == 0:
            return 1.0
        if len(ground_truth) == 0 or len(prediction) == 0:
            return 0.0

        edit_distance = levenshtein_distance(ground_truth, prediction)
        max_len = max(len(ground_truth), len(prediction))
        normalized_distance = edit_distance / max_len

        if normalized_distance < threshold:
            anls = 1 - normalized_distance
        else:
            anls = 0.0

        return anls
    except ImportError:
        from jiwer import cer
        return 1 - cer(ground_truth, prediction)


def calculate_all_metrics(ground_truth, prediction):
    """Calculate all metrics: CER, WER, BLEU, ANLS, METEOR"""
    metrics = {}

    # CER (Character Error Rate) - Lower is better
    # CLAMP to [0, 1] range since jiwer can return > 1.0
    # when prediction is much longer than reference
    try:
        raw_cer = cer(ground_truth, prediction)
        metrics['CER'] = min(raw_cer, 1.0)  # Cap at 1.0 (100% error)
    except:
        metrics['CER'] = 1.0

    # WER (Word Error Rate) - Lower is better
    # Same issue: WER can exceed 1.0
    try:
        raw_wer = wer(ground_truth, prediction)
        metrics['WER'] = min(raw_wer, 1.0)  # Cap at 1.0
    except:
        metrics['WER'] = 1.0

    # BLEU Score - Higher is better (0-1) — already bounded
    try:
        reference = [list(ground_truth)]
        hypothesis = list(prediction)
        smoothing = SmoothingFunction().method1
        raw_bleu = sentence_bleu(reference, hypothesis,
                                        smoothing_function=smoothing)
        metrics['BLEU'] = max(0.0, min(raw_bleu, 1.0))
    except:
        metrics['BLEU'] = 0.0

    # ANLS - Higher is better — already bounded
    try:
        raw_anls = calculate_anls(ground_truth, prediction)
        metrics['ANLS'] = max(0.0, min(raw_anls, 1.0))
    except:
        metrics['ANLS'] = 0.0

    # METEOR - Higher is better (0-1) — already bounded
    try:
        reference_tokens = list(ground_truth)
        hypothesis_tokens = list(prediction)
        raw_meteor = meteor_score([reference_tokens], hypothesis_tokens)
        metrics['METEOR'] = max(0.0, min(raw_meteor, 1.0))
    except:
        metrics['METEOR'] = 0.0

    # Character Accuracy — now guaranteed 0-100%
    metrics['Char_Accuracy'] = max((1 - metrics['CER']) * 100, 0.0)

    return metrics

In [ ]:
# ========================================
# CHECKPOINT MANAGEMENT
# ========================================

def save_checkpoint(all_results, year_results, last_processed_idx, output_dir):
    """Save checkpoint to resume later"""
    checkpoint_data = {
        'all_results': all_results,
        'year_results': dict(year_results),
        'last_processed_idx': last_processed_idx
    }

    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint_data, f)


def load_checkpoint(output_dir):
    """Load checkpoint if exists"""
    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            checkpoint_data = pickle.load(f)
        return checkpoint_data
    return None

In [ ]:
    # Save final checkpoint
    save_checkpoint(all_results, year_results, total_samples - 1, output_dir)
    print("\n[OK] All samples processed!")


In [ ]:
    # ========================================
    # 4. BEST AND WORST PERFORMING SAMPLES
    # ========================================
    print("\n" + "="*80)
    print("BEST PERFORMING SAMPLES (Top 5 by Character Accuracy)")
    print("="*80)

    best_samples = df.nlargest(5, 'Char_Accuracy')[display_cols]
    print("\n" + best_samples.to_string(index=False))

    print("\n" + "="*80)
    print("WORST PERFORMING SAMPLES (Bottom 5 by Character Accuracy)")
    print("="*80)

    worst_samples = df.nsmallest(5, 'Char_Accuracy')[display_cols]
    print("\n" + worst_samples.to_string(index=False))

    # ========================================
    # 5. SAVE RESULTS TO CSV
    # ========================================
    output_csv = "./test_results_tesseract/test_metrics.csv"
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\nFull results saved to: {output_csv}")

    year_csv = "./test_results_tesseract/year_wise_metrics.csv"
    year_df.to_csv(year_csv, index=False)
    print(f"Year-wise results saved to: {year_csv}")

    # ========================================
    # 6. SUMMARY STATISTICS
    # ========================================
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)

    print(f"\nSamples with >90% accuracy: {len(df[df['Char_Accuracy'] > 90])}")
    print(f"Samples with >80% accuracy: {len(df[df['Char_Accuracy'] > 80])}")
    print(f"Samples with <50% accuracy: {len(df[df['Char_Accuracy'] < 50])}")

    print(f"\nMedian Character Accuracy: {df['Char_Accuracy'].median():.2f}%")
    print(f"Std Dev Character Accuracy: {df['Char_Accuracy'].std():.2f}%")

    return df, year_df


In [ ]:
# ========================================
# STEP 7: RUN TESTING
# ========================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("STARTING TESSERACT 5.x OCR BENCHMARK")
    print("="*80)
    print(f"Tesseract Version: {pytesseract.get_tesseract_version()}")
    print(f"Language: {TESSERACT_LANG}")
    print(f"Config: {TESSERACT_CONFIG}")

    try:
        # Run testing with resume support
        all_results, year_results = test_entire_dataset(resume=True)

        # Display results
        results_df, year_df = display_results(all_results, year_results)

        print("\n" + "="*80)
        print("TESTING COMPLETED!")
        print("="*80)
        print(f"\nCheck ./test_results_tesseract/ directory for:")
        print(f"  - test_metrics.csv (all samples)")
        print(f"  - year_wise_metrics.csv (year summary)")
        print(f"  - inference_outputs/ (Tesseract predictions as .txt files)")
        print(f"  - checkpoint.pkl (resume checkpoint)")

    except KeyboardInterrupt:
        print("\n\nTesting paused. Run again to resume from checkpoint.")


In [ ]:
# ========================================
# STEP 8: ZIP & DOWNLOAD RESULTS
# ========================================

import shutil
import os
from IPython.display import FileLink

# Create zip file of test_results folder
output_zip = 'test_results_tesseract.zip'

if os.path.exists('./test_results_tesseract'):
    # Remove existing zip if it exists
    if os.path.exists(output_zip):
        os.remove(output_zip)

    # Create zip archive
    shutil.make_archive('test_results_tesseract', 'zip', './test_results_tesseract')

    print(f"[OK] Created {output_zip}")
    print(f"Size: {os.path.getsize(output_zip) / (1024*1024):.2f} MB")

    # Provide download link
    display(FileLink(output_zip))
else:
    print("[ERROR] test_results_tesseract folder not found")
